# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Álvaro Sifuentes Tasayco
- Nombre de alumno 2: Sebastián Morales Castillo

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.14.2 environment at: C:\Users\LENOVO\Documents\MDS7202\.venv
Checked 8 packages in 176ms


In [2]:
import time
from concurrent.futures import ThreadPoolExecutor  # noqa: F401  TODO: usar en load_all_parallel (1.2)
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [3]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [4]:
# Escribe aquí tu código
df = load_all_serial(DATA_DIR)

# Exploración inicial
print("Shape del DataFrame:", df.shape)
print("\nTipos de datos originales:")
display(df.dtypes)

print("\nUso de memoria por columna antes de optimizar:")
display(df.memory_usage(deep=True).sort_values(ascending=False) / 1024**2)

df_opt = df.copy()  # .astype(...)
# Columnas a convertir
float_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "avg_artist_popularity",
]

int16_cols = [
    "key",
    "mode",
]

int32_cols = [
    "year",
    "popularity",
    "duration_ms",
    "total_artist_followers",
]

# Conversión de tipos
df_opt[float_cols] = df_opt[float_cols].astype("float32")
df_opt[int16_cols] = df_opt[int16_cols].astype("int16")
df_opt[int32_cols] = df_opt[int32_cols].astype("int32")

# Comparación de memoria
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2
mem_reduction_mib = mem_before - mem_after
mem_reduction_pct = (1 - mem_after / mem_before) * 100

print(f"Memoria antes: {mem_before:.2f} MiB")
print(f"Memoria después: {mem_after:.2f} MiB")
print(f"Reducción: {mem_reduction_mib:.2f} MiB")
print(f"Reducción porcentual: {mem_reduction_pct:.2f}%")

print("\nTipos de datos después de optimizar:")
display(df_opt.dtypes)
# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

Shape del DataFrame: (200000, 24)

Tipos de datos originales:


id                         object
name                       object
album_name                 object
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                     object
year                        int64
genre                      object
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object


Uso de memoria por columna antes de optimizar:


lyrics                    271.049740
niche_genres               22.888184
artist_ids                 22.888184
artists                    22.888184
id                         13.542175
album_name                 13.233352
name                       13.006796
genre                      10.337770
danceability                1.525879
duration_ms                 1.525879
loudness                    1.525879
energy                      1.525879
key                         1.525879
mode                        1.525879
speechiness                 1.525879
acousticness                1.525879
instrumentalness            1.525879
popularity                  1.525879
tempo                       1.525879
valence                     1.525879
liveness                    1.525879
avg_artist_popularity       1.525879
year                        1.525879
total_artist_followers      1.525879
Index                       0.000126
dtype: float64

Memoria antes: 414.25 MiB
Memoria después: 401.28 MiB
Reducción: 12.97 MiB
Reducción porcentual: 3.13%

Tipos de datos después de optimizar:


id                         object
name                       object
album_name                 object
artists                    object
danceability              float32
energy                    float32
key                         int16
loudness                  float32
mode                        int16
speechiness               float32
acousticness              float32
instrumentalness          float32
liveness                  float32
valence                   float32
tempo                     float32
duration_ms                 int32
lyrics                     object
year                        int32
genre                      object
popularity                  int32
total_artist_followers      int32
avg_artist_popularity     float32
artist_ids                 object
niche_genres               object
dtype: object

### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

Respuestas:

1- Parquet es un formato de almacenamiento de datos pensado para análisis, que guarda la información de forma columnar y comprimida. A diferencia de CSV, no almacena todo como texto plano, sino que conserva mejor los tipos de datos y permite leer solo las columnas necesarias. Esto es una ventaja en datos analíticos, porque muchas veces no se necesita cargar todo el dataset, sino solo ciertas variables. El columnar storage significa que los datos se organizan por columnas en vez de por filas, lo que acelera las consultas porque el sistema puede leer únicamente las columnas requeridas y evitar cargar información innecesaria.

2- Apache Arrow es una tecnología que define una forma eficiente de representar datos en memoria usando un formato columnar. Se relaciona con Parquet porque ambos están pensados para trabajar eficientemente con datos por columnas: Parquet se usa principalmente para almacenar datos en disco, mientras que Arrow facilita su uso eficiente en memoria. Pandas puede usar motores como PyArrow para leer archivos Parquet de manera rápida. Al usar pd.read_parquet en vez de pd.read_csv, se gana velocidad de lectura, mejor conservación de tipos de datos, compresión y una carga más eficiente para datasets grandes.

3- Float32 existe porque usa menos memoria que float64. Mientras float64 ocupa 64 bits, float32 ocupa solo 32 bits, por lo que permite reducir el uso de memoria en columnas numéricas grandes. Aunque float64 es más preciso, esa precisión extra no siempre es necesaria. En contextos como machine learning, sensores, datos de audio o variables continuas aproximadas, pequeñas diferencias decimales suelen ser irrelevantes para el análisis o el entrenamiento del modelo. En este caso, columnas como danceability, energy, tempo o valence pueden almacenarse como float32 sin que eso afecte de manera importante el resultado.

4- No conviene reducir la precisión cuando se trabaja con datos que requieren alta exactitud, como cálculos financieros, coordenadas geográficas muy precisas, mediciones científicas sensibles, identificadores numéricos o acumulaciones donde pequeños errores pueden amplificarse. Los riesgos concretos son perder información por redondeo, cambiar ligeramente los valores originales, generar errores acumulados o incluso producir overflow si se usa un tipo entero demasiado pequeño para los valores reales. Por eso, antes de convertir tipos, siempre hay que revisar el rango de valores y la importancia de la precisión para el problema.

5- Sí, existen alternativas a pandas que pueden ser más eficientes en memoria y rendimiento. Dos ejemplos son Polars y DuckDB. Polars está implementado en Rust, trabaja con un modelo columnar basado en Apache Arrow y suele ser muy rápido para operaciones analíticas. DuckDB permite consultar datos con SQL directamente desde archivos como Parquet, sin necesidad de cargar todo el dataset completo en memoria. También se puede mencionar PyArrow, especialmente cuando se trabaja con datos columnares y archivos Parquet.

6- En la ejecución, el uso de memoria se redujo de 414.2 MiB a 401.3 MiB, lo que equivale a una reducción aproximada de 12.9 MiB, es decir, cerca de un 3.1%. Este resultado era esperable porque se optimizaron varias columnas numéricas, cambiando float64 a float32 e int64 a enteros más pequeños como int16 o int32. Sin embargo, la reducción no fue tan grande porque el dataset contiene varias columnas de tipo object, como id, name, album_name, artists, lyrics, genre, artist_ids y niche_genres. Estas columnas de texto ocupan gran parte de la memoria total y no fueron optimizadas en esta parte, por lo que el impacto de cambiar solo las columnas numéricas fue relativamente limitado.

7- Si se redujera valence a float16, se usaría aún menos memoria, pero también se perdería más precisión en la variable objetivo. Esto puede ser problemático porque valence es justamente la variable que se quiere predecir en la sección 2. Al reducirla demasiado, podrían alterarse diferencias pequeñas entre canciones, haciendo que el modelo aprenda desde una señal menos precisa. El riesgo sería obtener un entrenamiento menos representativo y métricas que no reflejen tan bien el comportamiento real de la variable original.

In [5]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [6]:
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en paralelo y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    with ThreadPoolExecutor() as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


print("Serial:")
%timeit -n 1 -r 3 load_all_serial(DATA_DIR)
print("Paralelo:")
%timeit -n 1 -r 3 load_all_parallel(DATA_DIR)

Serial:
1.75 s ± 27.7 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)
Paralelo:
1.26 s ± 66.9 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [7]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?

    Una operación I/O-bound pasa la mayor parte del tiempo esperando operaciones externas (disco, red, DB), mientras que una CPU-bound pasa la mayor parte computando. Leer archivos desde disco es I/O-bound: el CPU está bloqueado esperando los datos del disco.

2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?

    Es un mutex global de CPython que permite que solo un hilo ejecute bytecode de Python a la vez. Existe porque la gestión de memoria por reference counting no es thread-safe y un lock global es más simple que poner locks finos en cada operación. Resuelve la corrupción del intérprete bajo concurrencia, pero impide que dos hilos ejecuten código Python en paralelo sobre varios cores.

3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?

    Por la productividad y el ecosistema. Las librerías pesadas (NumPy, Arrow, PyTorch, polars) están escritas en C/C++/Rust y liberan el GIL durante sus operaciones, así que Python orquesta sin pagar su costo. Lo mismo pasa con I/O, ya que las syscalls liberan el GIL mientras esperan, por eso los threads sí ayudan en este lab.

4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?

    ThreadPoolExecutor para tareas I/O-bound (los hilos liberan el GIL durante la espera y comparten memoria con poco overhead). ProcessPoolExecutor para tareas CPU-bound, ya que cada proceso tiene su propio GIL y puede usar un core completo. Si fuera puramente CPU-bound, usaría ProcessPoolExecutor.

5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?

    Arrancar los hilos, mantener la cola de tareas, sincronizar con locks y devolver resultados. Con archivos de 1 KB la lectura toma microsegundos y el overhead del pool supera al trabajo útil, por lo que la versión serial sería igual o más rápida. La solución sería agrupar los archivos en lotes más grandes antes de paralelizar.

6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?

    Sí. Con `%timeit` sobre los 20 archivos: serial $\approx$ 1.41 s y paralelo $\approx$ 0.74 s, un speedup de $\approx$ 1.9 $\times$. En el gráfico, las curvas son casi idénticas con 2-3 archivos (el overhead domina) y empiezan a separarse de forma clara a partir de $\sim$ 4 archivos.

7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

    Aunque hay 8 cores y 12 hilos en el pool, el speedup fue $\approx$ 1.9 $\times$. Lo limitan la parte serial irreducible (`pd.concat`, asignación de memoria final) que pone un techo según la ley de Amdahl, el parseo del Parquet en Python que compite por el GIL, el ancho de banda del disco que se satura con pocos hilos, el overhead de threading y la asignación de memoria que también se serializa parcialmente.

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [8]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [10]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params))  # noqa: B905
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [11]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

In [12]:
print("Versión de pandas:", pd.__version__)
print("Versión de Polars:", pl.__version__)

max_iter = df_bench["iteration"].max()
df_last = df_bench[df_bench["iteration"] == max_iter].sort_values("time_took")

print(f"\nResultados para {max_iter} filas:")
display(df_last)

fastest = df_last.iloc[0]
print(f"Implementación más rápida: {fastest['version']} con {fastest['time_took']:.6f} segundos")

pivot_times = df_bench.pivot(index="iteration", columns="version", values="time_took")

speedup_table = pivot_times.copy()
speedup_table["speedup_numpy"] = speedup_table["Python"] / speedup_table["NumPy"]
speedup_table["speedup_numba"] = speedup_table["Python"] / speedup_table["Numba-JIT"]
speedup_table["speedup_pandas"] = speedup_table["Python"] / speedup_table["Pandas"]
speedup_table["speedup_polars"] = speedup_table["Python"] / speedup_table["Polars"]

display(speedup_table.tail())

print("\nÚltimos speedups:")
display(speedup_table[["speedup_numpy", "speedup_numba", "speedup_pandas", "speedup_polars"]].tail())

Versión de pandas: 2.3.3
Versión de Polars: 1.41.2

Resultados para 39001 filas:


,time_took,iteration,version
227,0.000258,39001,Numba-JIT
229,0.000984,39001,Polars
226,0.002266,39001,NumPy
228,0.002333,39001,Pandas
225,0.149542,39001,Python


Implementación más rápida: Numba-JIT con 0.000258 segundos


version,NumPy,Numba-JIT,Pandas,Polars,Python,speedup_numpy,speedup_numba,speedup_pandas,speedup_polars
iteration,,,,,,,,,
35001,0.001952,0.000295,0.002515,0.001147,0.142875,73.190412,484.486211,56.806888,124.596669
36001,0.002030,0.000284,0.002755,0.001406,0.144086,70.992167,507.344102,52.292117,102.508316
37001,0.002500,0.000618,0.002903,0.001082,0.115943,46.371796,187.701783,39.943293,107.166458
38001,0.002044,0.000286,0.002443,0.001088,0.117459,57.462454,410.408852,48.077852,108.008274
39001,0.002266,0.000258,0.002333,0.000984,0.149542,65.993865,578.723062,64.101374,151.927371



Últimos speedups:


version,speedup_numpy,speedup_numba,speedup_pandas,speedup_polars
iteration,,,,
35001,73.190412,484.486211,56.806888,124.596669
36001,70.992167,507.344102,52.292117,102.508316
37001,46.371796,187.701783,39.943293,107.166458
38001,57.462454,410.408852,48.077852,108.008274
39001,65.993865,578.723062,64.101374,151.927371


### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

Respuestas:
1- La vectorización en NumPy consiste en aplicar operaciones directamente sobre arrays completos, sin tener que escribir loops explícitos en Python. Aunque internamente sí se recorren los datos, esas operaciones se ejecutan en código optimizado de bajo nivel, principalmente en C, evitando el costo de interpretar una instrucción de Python por cada fila o elemento. Por eso, en este benchmark NumPy fue mucho más rápido que Python puro, especialmente cuando aumentó el número de filas.

2- JIT significa Just-In-Time compilation, es decir, compilación justo a tiempo. En este caso, Numba toma una función escrita en Python y la compila a código máquina cuando se ejecuta por primera vez. El decorador @numba.njit intenta compilar la función en modo nopython, lo que significa que la función no depende de objetos generales de Python durante su ejecución. Esto permite que loops numéricos simples, como el de la regresión lineal, se ejecuten de manera mucho más eficiente.

3- Numba es más lento en la primera ejecución porque necesita compilar la función antes de ejecutarla. Ese costo inicial se conoce como warm-up de JIT. Después de esa primera compilación, las ejecuciones siguientes son mucho más rápidas. En el gráfico se observa un tiempo muy alto al inicio para Numba-JIT, lo que corresponde justamente a ese costo de compilación inicial, por lo que al analizar el rendimiento real conviene mirar las ejecuciones posteriores y no solo la primera medición.

4- Polars es una librería para trabajar con datos tabulares, similar a pandas, pero diseñada con foco en rendimiento y eficiencia de memoria. Está implementada en Rust, usa un modelo columnar basado en Apache Arrow y puede ejecutar operaciones de forma paralela. Fue pensada para escenarios analíticos con datasets medianos o grandes, donde importa mucho la velocidad de procesamiento. En este benchmark, Polars tuvo muy buen rendimiento y fue más rápido que pandas y NumPy en la medición final.

5- Polars se diferencia de pandas principalmente en su implementación y modelo de ejecución. Pandas está construido sobre NumPy y tiene una ejecución más tradicional, mientras que Polars está implementado en Rust y usa Apache Arrow como base de memoria columnar. Además, Polars permite ejecución lazy, optimización de consultas y paralelización interna en varias operaciones. Esto puede hacerlo más eficiente en memoria y más rápido que pandas en ciertos flujos de procesamiento de datos.

6- Pandas puede ser más lento que NumPy porque agrega una capa extra de abstracción sobre los datos. Un DataFrame maneja índices, nombres de columnas, alineamiento de datos, tipos mixtos y validaciones adicionales. Todo eso hace que pandas sea más cómodo para análisis de datos, pero también introduce overhead. NumPy, en cambio, trabaja directamente con arrays numéricos más simples, por lo que suele ser más eficiente en operaciones matemáticas puras.

7- Las instrucciones SIMD, Single Instruction Multiple Data, permiten aplicar una misma operación sobre varios datos al mismo tiempo. En vez de procesar un valor por instrucción, el procesador puede trabajar con bloques de valores simultáneamente. Esto acelera operaciones numéricas repetitivas sobre arrays o columnas, que es justamente el tipo de trabajo que hacen NumPy y Polars. Por eso estas librerías pueden ser mucho más rápidas que loops puros de Python.

8- Conviene usar Numba sobre NumPy cuando se tienen loops numéricos que son difíciles de vectorizar de forma clara o cuando una solución vectorizada generaría demasiados arreglos intermedios. Si el cálculo se puede expresar fácilmente como operaciones matriciales, NumPy suele ser suficiente. Polars conviene sobre pandas cuando se trabaja con datasets grandes, operaciones columnares, filtros, agregaciones o transformaciones encadenadas, especialmente si se quiere aprovechar mejor la memoria y la paralelización.

9- En mi medición, la implementación más rápida para 39001 filas fue Numba-JIT, con un tiempo de 0.000258 segundos. Esto era esperable después del warm-up, porque Numba compila el loop numérico a código máquina y evita el costo del intérprete de Python. También era esperable que Python puro fuera la alternativa más lenta, ya que procesó las filas directamente con loops interpretados. En la medición final, Python tardó 0.149542 segundos, mientras que Numba-JIT logró un speedup aproximado de 578.7 veces.

10- En la medición final no se observó una diferencia muy grande entre pandas y NumPy, ya que NumPy tardó 0.002266 segundos y pandas 0.002333 segundos para 39001 filas. NumPy fue levemente más rápido, lo cual tiene sentido porque trabaja directamente con arrays numéricos, mientras que pandas agrega overhead por manejar índices, nombres de columnas y alineamiento. Aun así, pandas tuvo un rendimiento bastante competitivo porque usa internamente operaciones vectorizadas optimizadas.

11- La ventaja de NumPy y Numba sobre Python puro empieza a ser evidente desde cantidades bajas de filas, pero se vuelve mucho más clara a medida que aumenta el tamaño del input. En el gráfico logarítmico se observa que Python puro crece de manera más marcada, mientras que NumPy y especialmente Numba se mantienen con tiempos mucho más bajos. A partir de algunos miles de filas, la diferencia ya es notoria, y para 39001 filas NumPy fue aproximadamente 66 veces más rápido que Python puro, mientras que Numba-JIT fue aproximadamente 579 veces más rápido.

12- Sí, en mi medición Polars fue más eficiente que pandas. Para 39001 filas, Polars tardó 0.000984 segundos, mientras que pandas tardó 0.002333 segundos, por lo que Polars fue más de dos veces más rápido en esa medición final. La versión de pandas utilizada fue 2.3.3 y la versión de Polars fue 1.41.2. La versión sí puede influir, porque pandas ha mejorado su rendimiento en el tiempo, especialmente en integración con Arrow y optimización de algunas operaciones. Aun así, Polars suele tener ventajas en pipelines analíticos grandes por su implementación en Rust, su modelo columnar y su paralelización interna.

13- Numba puede igualar o superar a NumPy en loops numéricos simples porque compila el loop a código máquina y evita el overhead del intérprete de Python. Además, en algunos casos NumPy crea arreglos intermedios para resolver una operación vectorizada, mientras que Numba puede ejecutar el loop directamente de forma eficiente. En este benchmark, Numba-JIT fue la opción más rápida después de la compilación inicial, lo que muestra que para este tipo de cálculo numérico simple puede superar a alternativas vectorizadas.

14- Si el benchmark incluyera el costo de convertir los datos a NumPy o Polars, los resultados podrían cambiar, especialmente para tamaños pequeños o medianos. La conversión tiene un costo adicional, por lo que pandas podría verse relativamente más competitivo si los datos ya están originalmente en un DataFrame. En producción, ese costo no existiría o sería menor si el pipeline ya trabajara desde el inicio con el formato final, por ejemplo arrays NumPy, tablas Arrow o DataFrames Polars.

15- Para realizar esta predicción sobre 100 millones de filas, elegiría una implementación basada en Numba-JIT o Polars, dependiendo del flujo completo de datos. Si la tarea fuera solamente aplicar la fórmula numérica sobre arrays ya preparados, usaría Numba-JIT porque en esta medición fue la implementación más rápida. Si además tuviera que cargar, filtrar y transformar datos tabulares, usaría Polars porque está diseñado para trabajar eficientemente con datos columnares grandes. Si dispusiera de una GPU, evaluaría cambiar a una implementación compatible con GPU, ya que ese volumen de datos podría beneficiarse de paralelismo masivo.

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [14]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 298.7s | RMSE: 0.1652
n_jobs=-1 → tiempo: 109.3s | RMSE: 0.1652


In [15]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

In [16]:
# Cálculo explícito de mejora para responder preguntas
speedup_training = time_1job / time_all_jobs
time_reduction = time_1job - time_all_jobs
time_reduction_pct = (1 - time_all_jobs / time_1job) * 100
rmse_difference = abs(rmse_1 - rmse_all)

print(f"Tiempo n_jobs=1: {time_1job:.2f} segundos")
print(f"Tiempo n_jobs=-1: {time_all_jobs:.2f} segundos")
print(f"Mejora absoluta: {time_reduction:.2f} segundos")
print(f"Reducción porcentual del tiempo: {time_reduction_pct:.2f}%")
print(f"Speedup: {speedup_training:.2f} veces")
print(f"RMSE n_jobs=1: {rmse_1:.4f}")
print(f"RMSE n_jobs=-1: {rmse_all:.4f}")
print(f"Diferencia de RMSE: {rmse_difference:.6f}")

Tiempo n_jobs=1: 298.75 segundos
Tiempo n_jobs=-1: 109.32 segundos
Mejora absoluta: 189.43 segundos
Reducción porcentual del tiempo: 63.41%
Speedup: 2.73 veces
RMSE n_jobs=1: 0.1652
RMSE n_jobs=-1: 0.1652
Diferencia de RMSE: 0.000000


### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

Respuestas:
1- El parámetro n_jobs controla cuántos recursos de procesamiento se utilizan para ejecutar ciertas tareas en paralelo. En el caso de RandomForestRegressor, esto es especialmente útil porque el modelo construye muchos árboles de decisión, y varios de ellos pueden entrenarse de manera independiente. Cuando se usa n_jobs=1, el entrenamiento se realiza usando un solo núcleo o trabajo. En cambio, cuando se usa n_jobs=-1, scikit-learn intenta utilizar todos los núcleos disponibles de la máquina, lo que puede reducir considerablemente el tiempo de entrenamiento.

2- Aquí sí funciona el paralelismo real porque el entrenamiento de un Random Forest se puede dividir naturalmente entre varios árboles, ya que cada árbol del bosque puede entrenarse de forma relativamente independiente. Además, scikit-learn utiliza herramientas como joblib y código optimizado en librerías de bajo nivel, por lo que gran parte del trabajo no depende directamente de loops interpretados en Python puro. Esto permite aprovechar varios núcleos de la máquina y reducir el impacto del GIL, que normalmente limita la ejecución paralela de código Python puro.

3- En mi ejecución, el entrenamiento con n_jobs=1 demoró 298.75 segundos, mientras que con n_jobs=-1 demoró 109.32 segundos. Esto significa que usar todos los núcleos disponibles redujo el tiempo de entrenamiento en 189.43 segundos, equivalente a una reducción aproximada del 63.41%. En términos de speedup, el entrenamiento con n_jobs=-1 fue aproximadamente 2.73 veces más rápido que con n_jobs=1, lo que muestra una mejora importante en el tiempo de entrenamiento.

4- La mejora no fue necesariamente proporcional al número de CPUs disponibles en la máquina, y eso es esperable. Aunque n_jobs=-1 permite paralelizar parte del entrenamiento, no todo el proceso puede dividirse perfectamente entre núcleos. También existen costos asociados a coordinar los trabajos paralelos, mover datos en memoria, construir el pipeline, aplicar el OneHotEncoder y juntar los resultados del modelo. Por eso, aunque el entrenamiento fue mucho más rápido, el aumento de velocidad real no crece de forma perfectamente lineal con la cantidad de CPUs disponibles.

5- No hubo diferencia en el RMSE entre ambas versiones. En mi ejecución, tanto el modelo con n_jobs=1 como el modelo con n_jobs=-1 obtuvieron un RMSE de 0.1652, con una diferencia de 0.000000. Esto era esperable, porque n_jobs solo cambia la forma en que se distribuye el trabajo computacional durante el entrenamiento, pero no cambia los datos, el modelo ni sus hiperparámetros. Además, como se usa random_state=42, ambos entrenamientos deberían ser reproducibles y entregar el mismo desempeño o uno prácticamente idéntico.

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [ ]:
# Escribe aquí tu código (copia el template y completa los TODOs)


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>